# Tahap 4 - Labeling Sentimen Menggunakan InSet Lexicon

Notebook ini digunakan untuk dokumentasi proses labeling sentimen pada project skripsi **Analisis Sentimen pada Aplikasi JakOne Mobile Menggunakan Metode IndoBERT**.

Tahap ini hanya membuat label sentimen berbasis InSet Lexicon. Notebook ini tidak melakukan split dataset, training IndoBERT, evaluasi model, atau visualisasi akhir project.

## Input dan Output

Input dataset hasil preprocessing:

```text
data/processed/jakone_reviews_clean.csv
```

Input InSet Lexicon:

```text
data/lexicon/positive.tsv
data/lexicon/negative.tsv
```

Output tahap labeling:

```text
data/processed/jakone_reviews_labeled.csv
data/processed/lexicon_validation_sample.csv
```

## Aturan Skor dan Label

Skor dihitung dari token pada kolom `clean_review`. Setiap token dicek pada dictionary positif dan negatif. Jika token sebelumnya adalah kata negasi seperti `tidak`, `belum`, atau `jangan`, maka bobot token sentimen berikutnya dibalik.

Aturan label:

```text
skor > 0  -> positif
skor < 0  -> negatif
skor = 0  -> netral
```

In [ ]:
from pathlib import Path
import importlib.util

import matplotlib.pyplot as plt
import pandas as pd

## Memuat Fungsi dari Script

Notebook ini menggunakan fungsi dari `src/04_labeling_inset.py` agar hasil labeling sama dengan ketika script dijalankan lewat terminal.

In [ ]:
module_path = Path("src/04_labeling_inset.py")
if not module_path.exists():
    module_path = Path("../src/04_labeling_inset.py")

spec = importlib.util.spec_from_file_location("labeling_module", module_path)
labeling_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(labeling_module)

## Membaca Dataset dan Lexicon

Dataset clean dibaca dari tahap preprocessing. File lexicon dibaca dengan separator tab, lalu kolom `weight` dikonversi menjadi numeric/float.

In [ ]:
labeling_module.validate_input_files()

df_clean = labeling_module.load_dataset(labeling_module.DATA_PATH)
positive_df = labeling_module.load_lexicon(labeling_module.POSITIVE_LEXICON_PATH, "positif")
negative_df = labeling_module.load_lexicon(labeling_module.NEGATIVE_LEXICON_PATH, "negatif")

positive_dict, negative_dict = labeling_module.build_lexicon_dictionaries(positive_df, negative_df)

print(f"Jumlah data clean: {len(df_clean)}")
print(f"Jumlah kata positif: {len(positive_dict)}")
print(f"Jumlah kata negatif: {len(negative_dict)}")

## Contoh Perhitungan Skor

Contoh ini memperlihatkan aturan negasi sederhana. Frasa `tidak bagus` membalik bobot kata `bagus` menjadi negatif.

In [ ]:
example_text = "tidak bagus tapi error"
example_score = labeling_module.calculate_sentiment_score(example_text, positive_dict, negative_dict)
example_label = labeling_module.assign_label(example_score)

print("Teks:", example_text)
print("Skor:", example_score)
print("Label:", example_label)

## Menjalankan Labeling

Setiap ulasan diberi kolom `lexicon_score` dan `label`. Output mempertahankan kolom lama dari dataset preprocessing.

In [ ]:
df_labeled = labeling_module.label_reviews(df_clean, positive_dict, negative_dict)
df_labeled[["review", "clean_review", "lexicon_score", "label"]].head(10)

## Distribusi Label

Distribusi label ditampilkan dalam jumlah dan persentase untuk melihat komposisi hasil labeling lexicon.

In [ ]:
label_counts = df_labeled["label"].value_counts().reindex(["positif", "negatif", "netral"], fill_value=0)
label_percentages = (label_counts / len(df_labeled) * 100).round(2)

display(pd.DataFrame({"jumlah": label_counts, "persentase": label_percentages}))

## Visualisasi Distribusi Label

Grafik berikut hanya digunakan sebagai dokumentasi distribusi label pada notebook tahap labeling.

In [ ]:
plt.figure(figsize=(7, 4))
bars = plt.bar(label_counts.index, label_counts.values, color=["#2e7d32", "#c62828", "#616161"])
plt.title("Distribusi Label Sentimen InSet Lexicon")
plt.xlabel("Label")
plt.ylabel("Jumlah Ulasan")

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height, f"{int(height)}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

## Contoh Data per Kelas

Bagian ini menampilkan tiga contoh ulasan untuk masing-masing label.

In [ ]:
for label in ["positif", "negatif", "netral"]:
    print(f"\nLabel: {label}")
    display(df_labeled[df_labeled["label"] == label][["clean_review", "lexicon_score", "label"]].head(3))

## Membuat Sample Validasi Manual

Sample validasi manual berisi 100 data acak dengan `random_state=42`. Kolom `manual_label` dan `notes` dikosongkan untuk diisi saat pemeriksaan manual.

In [ ]:
validation_sample = labeling_module.create_validation_sample(df_labeled)
validation_sample.head()

## Menyimpan Output

Dataset berlabel dan sample validasi manual disimpan ke folder `data/processed/`.

In [ ]:
labeling_module.OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_labeled.to_csv(labeling_module.OUTPUT_PATH, index=False, encoding="utf-8-sig")
validation_sample.to_csv(labeling_module.VALIDATION_SAMPLE_PATH, index=False, encoding="utf-8-sig")

print(f"Output utama: {labeling_module.OUTPUT_PATH}")
print(f"Output validasi manual: {labeling_module.VALIDATION_SAMPLE_PATH}")